# Brand Analysis

This notebook analyzes the cleaned and normalized fragrance brand data using `brand_clean` as the primary analytical dimension.

The analysis focuses on the representation of brands across sellers and their price positioning in the marketplace.

## Objectives

* Analyze the number of listings and sellers represented by each brand
* Identify brands available from the largest number of sellers
* Compare minimum and maximum prices across brands
* Identify brands with the widest price ranges
* Compare average and median selling prices across brands
* Examine the relationship between brand availability and price positioning
* Assess brand concentration within the marketplace
* Identify potential data-quality issues that may require further normalization

> **Note:** `gender` and `title` will be cleaned separately before analyses that depend on these fields.

In [0]:
# Brand Analysis — load cleaned data and create brand-level summary

from pyspark.sql import functions as F

# Load the cleaned marketplace dataset
df = spark.table("main.fragrance_analytics.marketplace_brand_clean")

# Create brand-level analytical summary
brand_summary = (
    df
    .filter(F.col("brand_clean").isNotNull())
    .groupBy("brand_clean")
    .agg(
        F.count("*").alias("listing_count"),
        F.countDistinct("seller").alias("seller_count"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.avg("price").alias("avg_price"),
        F.expr("percentile_approx(price, 0.5)").alias("median_price")
    )
    .withColumn(
        "price_range",
        F.col("max_price") - F.col("min_price")
    )
    .select(
        "brand_clean",
        "listing_count",
        "seller_count",
        "min_price",
        "max_price",
        "price_range",
        "avg_price",
        "median_price"
    )
    .orderBy(F.desc("listing_count"))
)

# Display the brand summary
display(brand_summary)

In [0]:
# Top brands by marketplace representation

top_brands_by_listings = (
    brand_summary
    .select(
        "brand_clean",
        "listing_count",
        "seller_count"
    )
    .orderBy(
        F.desc("listing_count"),
        F.desc("seller_count")
    )
    .limit(20)
)

display(top_brands_by_listings)

In [0]:
# Top brands by number of sellers

top_brands_by_sellers = (
    brand_summary
    .select(
        "brand_clean",
        "seller_count",
        "listing_count"
    )
    .orderBy(
        F.desc("seller_count"),
        F.desc("listing_count")
    )
    .limit(20)
)

display(top_brands_by_sellers)

In [0]:
# Compare the Top 20 brands by listings and by seller count

top_20_listings = (
    brand_summary
    .select("brand_clean")
    .orderBy(F.desc("listing_count"), F.desc("seller_count"))
    .limit(20)
    .withColumn("top_by_listings", F.lit(True))
)

top_20_sellers = (
    brand_summary
    .select("brand_clean")
    .orderBy(F.desc("seller_count"), F.desc("listing_count"))
    .limit(20)
    .withColumn("top_by_sellers", F.lit(True))
)

top_20_comparison = (
    top_20_listings
    .join(top_20_sellers, on="brand_clean", how="full")
    .fillna(False, ["top_by_listings", "top_by_sellers"])
    .withColumn(
        "top_group",
        F.when(
            F.col("top_by_listings") & F.col("top_by_sellers"),
            "Both"
        )
        .when(
            F.col("top_by_listings"),
            "Listings only"
        )
        .otherwise("Sellers only")
    )
    .orderBy("top_group", "brand_clean")
)

display(top_20_comparison)

In [0]:
# Price positioning of the Top 20 brands by listings

top_brands_price = (
    brand_summary
    .select(
        "brand_clean",
        "listing_count",
        "seller_count",
        "min_price",
        "max_price",
        "price_range",
        "avg_price",
        "median_price"
    )
    .orderBy(
        F.desc("listing_count"),
        F.desc("seller_count")
    )
    .limit(20)
)

display(top_brands_price)

In [0]:
# Investigate high-price listings

high_price_listings = (
    df
    .filter(F.col("price") > 1000)
    .select(
        "brand_clean",
        "title",
        "seller",
        "price",
        "priceWithCurrency",
        "itemLocation"
    )
    .orderBy(F.desc("price"))
)

display(high_price_listings)